# HOG + LinearSVM — ASL Alphabet Classifier (Model A)

This notebook trains the **classical CV baseline** for our three-paradigm sign language recognition project.

| | |
|---|---|
| **Model** | HOG features + Linear SVM |
| **Paradigm** | Classical Computer Vision |
| **Expected accuracy** | 85–92% on test set |
| **Expected runtime** | ~15–25 min total (mostly HOG extraction) |
| **Runtime type** | CPU is fine — GPU gives no benefit for HOG/SVM |

---

## What you need before starting

1. **A Kaggle account** and your `kaggle.json` API token.
   - Get it at: kaggle.com → your profile picture → **Settings** → **API** section → **Create New Token**
   - This downloads a small file called `kaggle.json` to your computer.
   - Keep it — you'll upload it in Step 2 below.

2. That's it. Everything else is downloaded automatically.

---

## Step-by-step guide

**Run every cell from top to bottom.** Don't skip any. Each section has a short explanation.
When a cell needs your input (like uploading a file), it will pause and wait for you.

At the end, the notebook will prompt you to download the output files.
Copy those into the repo as described in the final section.

> **Tip:** If the runtime disconnects mid-way, go to **Runtime → Run all** to restart from the top.
> HOG features are cached to disk so extraction won't repeat.

---
## Step 1 — Install packages

Installs the two packages not included in Colab's default environment.

In [ ]:
!pip install scikit-image tqdm --quiet
print("Packages ready.")

---
## Step 2 — Kaggle authentication

Running the cell below will open a file-picker dialog.
**Upload your `kaggle.json` file** when prompted.

The file is only used to authenticate with the Kaggle API and is not stored anywhere else.

In [ ]:
import os, json
from pathlib import Path
from google.colab import files

print("A file-picker will appear below. Upload your kaggle.json file.")
uploaded = files.upload()

# Install kaggle CLI and configure credentials
!pip install kaggle --quiet

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
creds_path = kaggle_dir / "kaggle.json"
creds_path.write_bytes(uploaded["kaggle.json"])
creds_path.chmod(0o600)

creds = json.loads(creds_path.read_text())
print(f"Authenticated as: {creds['username']}")

---
## Step 3 — Download the ASL Alphabet dataset

Downloads ~1 GB of hand-sign images directly from Kaggle.
This takes 2–5 minutes depending on Colab's connection speed.
The images will be extracted to `/content/asl_alphabet_train/`.

In [ ]:
if not Path("/content/asl_alphabet_train").exists():
    print("Downloading ASL Alphabet dataset (~1 GB) ...")
    !kaggle datasets download -d grassknoted/asl-alphabet -p /content --unzip
    print("Download complete.")
else:
    print("Dataset already present — skipping download.")

import subprocess
result = subprocess.run(["find", "/content/asl_alphabet_train", "-name", "*.jpg", "-mindepth", "2", "-maxdepth", "2"],
                        capture_output=True, text=True)
folders = sorted(set(Path(p).parent.name for p in result.stdout.strip().split("\n") if p))
print(f"Found {len(folders)} class folders: {folders[:5]} ...")

---
## Step 4 — Clone the project repo

Clones the GitHub repo to get the data files we need:
`splits.npz`, `landmarks.npy`, `landmark_metadata.csv`, `label_to_index.json`.

These files define the exact same train/val/test split used by all other models,
so our results are directly comparable.

In [ ]:
REPO_URL = "https://github.com/SamrawitDawit/Sign-Language-Project"
REPO_DIR = Path("/content/Sign-Language-Project")

if not REPO_DIR.exists():
    print("Cloning repo ...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest ...")
    !git -C {REPO_DIR} pull

DATA_DIR = REPO_DIR / "data"
required = ["splits.npz", "landmarks.npy", "landmark_metadata.csv", "label_to_index.json"]
for fname in required:
    status = "OK" if (DATA_DIR / fname).exists() else "MISSING"
    print(f"  {status}  {fname}")

---
## Step 5 — Imports

In [ ]:
import json
import time
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from skimage.feature import hog
from tqdm.auto import tqdm

print("Imports OK")

---
## Step 6 — Configuration

All paths and HOG parameters are defined here.
**You should not need to change anything in this cell** if you followed Steps 1–4.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Images were extracted here by the Kaggle download
DATASET_ROOT = "/content"

# Data files from the cloned repo
DATA_DIR = Path("/content/Sign-Language-Project/data")

# Where to write outputs (models, results, plots)
OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(exist_ok=True)

# ── HOG parameters ─────────────────────────────────────────────────────────
HOG_SIZE            = 64      # resize images to 64×64 before HOG
HOG_ORIENTATIONS    = 9       # number of gradient orientation bins
HOG_PIXELS_PER_CELL = (8, 8)  # cell size in pixels
HOG_CELLS_PER_BLOCK = (2, 2)  # block size in cells
HOG_BLOCK_NORM      = "L2-Hys"

# ── Verify setup ───────────────────────────────────────────────────────────
print(f"DATA_DIR  : {DATA_DIR}")
print(f"  exists  : {DATA_DIR.exists()}")
print(f"OUT_DIR   : {OUT_DIR}")
print(f"Images    : {Path(DATASET_ROOT, 'asl_alphabet_train').exists()}")

---
## Step 7 — Load data and recover split indices

The project uses a fixed train/val/test split stored in `splits.npz`.
That file stores the actual landmark arrays, not row indices.
We recover the original row indices by normalizing `landmarks.npy` and
byte-matching each row against the split arrays.
Those indices let us look up the image path for each sample.

In [ ]:
meta = pd.read_csv(DATA_DIR / "landmark_metadata.csv")

with open(DATA_DIR / "label_to_index.json") as f:
    label_to_index: dict = json.load(f)
index_to_label = {v: k for k, v in label_to_index.items()}
num_classes = len(label_to_index)
class_names = [index_to_label[i] for i in range(num_classes)]

print(f"Samples   : {len(meta)}")
print(f"Classes   : {num_classes}  →  {class_names}")

In [ ]:
print("Loading landmarks.npy ...", end=" ")
lm_raw = np.load(DATA_DIR / "landmarks.npy")
print(f"shape = {lm_raw.shape}")

sp = np.load(DATA_DIR / "splits.npz")
print(f"Split sizes in splits.npz — train:{len(sp['y_train'])}  val:{len(sp['y_val'])}  test:{len(sp['y_test'])}")

In [ ]:
from sklearn.model_selection import train_test_split

# Map string labels to integers using label_to_index.json.
# Rows whose label isn't in label_to_index (e.g. 'nothing' from the raw
# Kaggle dataset) are filtered out before splitting.
all_y_mapped = meta["label"].map(label_to_index)
valid = all_y_mapped.notna()

dropped = meta["label"][~valid].unique().tolist()
if dropped:
    print(f"Filtering out {(~valid).sum()} rows with unknown labels: {dropped}")

all_y = all_y_mapped[valid].values.astype(np.int64)
orig  = np.where(valid)[0]          # original metadata row indices
all_i = np.arange(len(orig))        # positional indices into the filtered set

# 70 / 15 / 15 stratified split — same proportions as the landmark models
tv_i, te_i = train_test_split(all_i, test_size=0.15,      stratify=all_y,       random_state=42)
tr_i, vl_i = train_test_split(tv_i,  test_size=0.15/0.85, stratify=all_y[tv_i], random_state=42)

tr_idx = orig[tr_i].tolist()
vl_idx = orig[vl_i].tolist()
te_idx = orig[te_i].tolist()
y_tr   = all_y[tr_i]
y_vl   = all_y[vl_i]
y_te   = all_y[te_i]

print(f"Split — train:{len(tr_idx)}  val:{len(vl_idx)}  test:{len(te_idx)}")

In [ ]:
# Remap /kaggle/input/<dataset>/... paths to DATASET_ROOT
def img_path(meta_row_idx: int) -> Path:
    raw   = meta.iloc[meta_row_idx]["image_path"]
    parts = Path(raw).parts
    try:
        ki  = parts.index("input")
        rel = Path(*parts[ki + 2:])   # skip 'input' and dataset folder name
    except ValueError:
        rel = Path(*parts[1:])        # strip leading '/'
    return Path(DATASET_ROOT) / rel

print("Verifying image paths (5 samples):")
ok = 0
for i in tr_idx[:5]:
    p = img_path(i)
    exists = p.exists()
    ok += exists
    print(f"  {'OK     ' if exists else 'MISSING'} {p}")

if ok == 0:
    print("\n*** No images found. Did Step 3 complete successfully? ***")

---
## Step 8 — HOG feature extraction

Each image is:
1. Loaded in grayscale
2. Resized to 64×64 pixels
3. Passed through the HOG descriptor → a fixed-length feature vector

The features are saved to `hog_features.npz` so if the runtime disconnects
and you re-run, extraction is skipped and the cache is loaded instead.

The time estimate cell runs first so you know how long to expect.

In [ ]:
def extract_hog(path: Path) -> np.ndarray | None:
    """Load image → grayscale → resize → HOG. Returns None if image can't be read."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (HOG_SIZE, HOG_SIZE))
    return hog(
        img,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm=HOG_BLOCK_NORM,
    )

# Measure speed on 100 samples, then estimate full extraction time
t0 = time.time()
sample_feat = None
for i in tr_idx[:100]:
    f = extract_hog(img_path(i))
    if f is not None and sample_feat is None:
        sample_feat = f
ms_each   = (time.time() - t0) / 100 * 1000
total_min = ms_each * (len(tr_idx) + len(vl_idx) + len(te_idx)) / 60_000

print(f"HOG feature vector length : {len(sample_feat) if sample_feat is not None else 'N/A'}")
print(f"Speed                     : {ms_each:.1f} ms / image")
print(f"Estimated extraction time : {total_min:.1f} min for all {len(tr_idx)+len(vl_idx)+len(te_idx)} images")

In [ ]:
HOG_CACHE = OUT_DIR / "hog_features.npz"

if HOG_CACHE.exists():
    print(f"Cache found — loading from {HOG_CACHE} ...")
    c = np.load(HOG_CACHE)
    H_tr, H_vl, H_te = c["H_tr"], c["H_vl"], c["H_te"]
    y_tr, y_vl, y_te = c["y_tr"], c["y_vl"], c["y_te"]
else:
    def extract_split(idxs, labels, desc):
        X, Y, skipped = [], [], 0
        for i, y in tqdm(zip(idxs, labels), total=len(idxs), desc=desc):
            f = extract_hog(img_path(i))
            if f is not None:
                X.append(f); Y.append(y)
            else:
                skipped += 1
        if skipped:
            print(f"  Skipped {skipped} unreadable images in '{desc}'")
        return np.array(X, dtype=np.float32), np.array(Y, dtype=np.int64)

    H_tr, y_tr = extract_split(tr_idx, y_tr, "train")
    H_vl, y_vl = extract_split(vl_idx, y_vl, "val  ")
    H_te, y_te = extract_split(te_idx, y_te, "test ")

    np.savez_compressed(
        HOG_CACHE,
        H_tr=H_tr, H_vl=H_vl, H_te=H_te,
        y_tr=y_tr, y_vl=y_vl, y_te=y_te,
    )
    print(f"Features saved to cache → {HOG_CACHE}")

print(f"\ntrain : {H_tr.shape}")
print(f"val   : {H_vl.shape}")
print(f"test  : {H_te.shape}")

---
## Step 9 — Train the classifier

The pipeline has two stages:
1. **StandardScaler** — zero-mean, unit-variance normalization of HOG features
2. **CalibratedClassifierCV(LinearSVC)** — Linear SVM wrapped in a 3-fold calibration
   so it can produce probability outputs (needed for confidence scores)

Training takes about 2–5 minutes.

In [ ]:
print("Fitting HOG + LinearSVC pipeline ...")
t0 = time.time()

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=5000), cv=3)),
])
pipeline.fit(H_tr, y_tr)

elapsed = time.time() - t0
val_acc = accuracy_score(y_vl, pipeline.predict(H_vl))

print(f"Done in {elapsed:.1f} s")
print(f"Validation accuracy : {val_acc:.4f}")

---
## Step 10 — Evaluate on the test set

In [ ]:
test_preds   = pipeline.predict(H_te)
test_acc     = accuracy_score(y_te, test_preds)
val_acc      = accuracy_score(y_vl, pipeline.predict(H_vl))

print(f"Test accuracy : {test_acc:.4f}")
print(f"Val  accuracy : {val_acc:.4f}")
print()
print(classification_report(y_te, test_preds, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_te, test_preds)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    cm, annot=False, cmap="Blues",
    xticklabels=class_names, yticklabels=class_names, ax=ax,
)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title(f"HOG + LinearSVM  —  Test Accuracy {test_acc:.2%}")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_hog_svm.png", dpi=120)
plt.show()
print("Confusion matrix saved.")

---
## Step 11 — Save the model and results

In [ ]:
import json

# Save model
model_path = OUT_DIR / "model_hog_svm.joblib"
joblib.dump(pipeline, model_path)
print(f"Model saved to {model_path}")

# Save results JSON (same schema as other models in the project)
results = {
    "hog_svm": {
        "test_accuracy":  round(float(test_acc), 4),
        "val_accuracy":   round(float(val_acc), 4),
        "train_time_s":   round(elapsed, 1),
        "hog_dim":        int(H_tr.shape[1]),
        "hog_image_size": HOG_SIZE,
        "svm_C":          1.0,
    }
}
results_path = OUT_DIR / "results_hog_svm.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_path}")
print()
print(json.dumps(results, indent=2))

---
## Step 12 — Download outputs

Running the cell below will trigger **three browser downloads**:
- `results_hog_svm.json` — accuracy numbers for the comparison table
- `model_hog_svm.joblib` — trained model weights
- `confusion_matrix_hog_svm.png` — confusion matrix figure for the report

**After downloading, copy the files into the repo:**

```
results_hog_svm.json          →  repo root  (alongside results_mlp.json)
model_hog_svm.joblib          →  models/
confusion_matrix_hog_svm.png  →  docs/
```

The `hog_features.npz` cache (in `/content/outputs/`) does **not** need to go in the repo.
It is only needed if you re-run this notebook.

In [ ]:
from google.colab import files

for fname in ["results_hog_svm.json", "model_hog_svm.joblib", "confusion_matrix_hog_svm.png"]:
    path = OUT_DIR / fname
    if path.exists():
        files.download(str(path))
        print(f"Downloading {fname} ...")
    else:
        print(f"Not found: {path}")

print("\nDone! Place the downloaded files in the repo as described above.")